# Chat 3 — Module 3: Error Analysis & Misclassification Patterns

**Project:** Anemia CDS Pipeline — Clinical Decision Support System  
**Goal:** Systematically analyze the error structure of the two-stage model  
**Input:** M1 calibrated probabilities + M2 threshold/zone configuration  
**Output:** Error analysis tables, figures, clinical risk matrix, `src/m3_errors.py`

---
**Contents:**
1. Setup & Data Loading
2. 3A — Confusion Matrix & Per-Class Metrics
3. 3B — Misclassification Pattern Analysis
4. 3C — Clinical Error Impact Analysis
5. 3D — Feature Contribution to Errors
6. `src/m3_errors.py` Module Creation
7. Manifest Update

## 1. Setup & Data Loading

### 1.1 Drive Mount & Import Workaround

In [ ]:
# --- Drive Mount ---
# from google.colab import drive
# drive.mount('/content/drive')
# --- Import Workaround ---
import importlib, sys, os

src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

for mod_name in ['plot_style', 'cds_config', 'm1_calibration', 'm2_thresholds']:
    spec = importlib.util.spec_from_file_location(
        mod_name, os.path.join(src_dir, f"{mod_name}.py"))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)

from cds_config import *
setup_notebook("Chat 3 — M3 Error Analysis")
from m1_calibration import load_calibrated_probs
from m2_thresholds import (load_threshold_config, apply_stage1_threshold,
                           assign_confidence_zone, get_operating_point)

print("✅ Modules loaded")


### 1.2 Additional Libraries & Constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_fscore_support, accuracy_score)
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')

# --- Constants ---
S1_LABELS = {0: 'DAS', 1: 'IAS'}
S2_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}
S1_NAMES  = ['DAS', 'IAS']
S2_NAMES  = ['DEA', 'HA', 'HGB_HTZ', 'Normal']
# Display labels (for figure axes/titles; internal keys preserved)
DISP_MAP = {'DEA': 'IDA', 'HGB_HTZ': 'HGB HTZ', 'DAS': 'OAC', 'IAS': 'AAC'}
S1_DISPLAY = [DISP_MAP.get(c, c) for c in S1_NAMES]   # ['OAC', 'AAC']
S2_DISPLAY = [DISP_MAP.get(c, c) for c in S2_NAMES]   # ['IDA', 'HA', 'HGB HTZ', 'Normal']
# ── FEATURE DISPLAY MAPPING (for figure axes/titles — publication names) ──
# INTERNAL feature names (parquet columns) are preserved; translated only for DISPLAY.
FEATURE_BASE = {
    'mcv_f_l':'MCV','rdw_sd_fl':'RDW-SD','ret_he_pg':'Ret-He','mchc_g_dl':'MCHC',
    'hgb_g_d_l':'HGB','rbc_10_6_u_l':'RBC','irf_pct':'IRF','delta_he_pg':'Delta-He',
    'frc_perc':'FRC','uibc':'UIBC','ldh':'LD','ferritin':'Ferritin','demir':'Iron',
    'micro_macro_ratio':'MicroR/MacroR','ret_number_10_6_l':'RET#','yas':'Age',
}
# Terms that need parentheses inside a ratio (their display already contains '/')
_FEAT_NEEDS_PAREN = {'micro_macro_ratio'}

def _feat_disp_part(p, in_ratio=False):
    d = FEATURE_BASE.get(p, p)
    if in_ratio and p in _FEAT_NEEDS_PAREN:
        return f"({d})"
    return d

def feat_disp(name):
    """Convert an INTERNAL feature name to its publication-display name. X_div_Y → disp(X)/disp(Y)."""
    if name in FEATURE_BASE and '_div_' not in name:
        return FEATURE_BASE[name]
    if '_div_' in name:
        num, den = name.split('_div_', 1)
        return f"{_feat_disp_part(num, True)}/{_feat_disp_part(den, True)}"
    return name

CLASS_COLORS = {
    'DAS': PALETTE['base2'], 'IAS': PALETTE['accent3'],
    'DEA': PALETTE['highlight'], 'HA': PALETTE['base1'],
    'HGB_HTZ': PALETTE['accent1'], 'Normal': PALETTE['accent2']
}

ZONE_COLORS = {
    'HIGH': PALETTE['accent1'], 'MEDIUM': PALETTE['accent2'],
    'LOW': PALETTE['highlight']
}

# --- Directories ---
m3_dir = PATHS.module_dir('m3_errors')
analysis_dir = PATHS.module_dir('m3_errors', 'analysis')
plots_dir = PATHS.module_dir('m3_errors', 'plots')
reports_dir = PATHS.module_dir('m3_errors', 'reports')
for d in [m3_dir, analysis_dir, plots_dir, reports_dir]:
    os.makedirs(d, exist_ok=True)

print(f"📁 M3 directory: {m3_dir}")
print("✅ Constants and directories ready")

### 1.3 Data Loading

In [ ]:
# --- Load all data ---
data = {}
for scenario in SCENARIOS:
    data[scenario] = {}
    for stage in [1, 2]:
        data[scenario][stage] = {}
        for split in ['oof', 'test']:
            df = load_calibrated_probs(PATHS, stage, scenario, split)
            data[scenario][stage][split] = df
            print(f"  {scenario} S{stage} {split}: {df.shape}")

print("\n✅ All data loaded")

### 1.4 Threshold Config Loading & Prediction Generation

In [ ]:
th_config = load_threshold_config(PATHS)

# Inspect the threshold_config structure
import json

config_path = os.path.join(PATHS.module_dir('m2_thresholds', 'configs'), 'threshold_config.json')
with open(config_path, 'r') as f:
    raw = json.load(f)

# Show the full structure
print(json.dumps(raw, indent=2))

# Also: what does get_operating_point return?
for sc in SCENARIOS:
    op = get_operating_point(th_config, sc)
    print(f"\n{sc} keys: {list(op.keys())}")
    print(f"{sc} values: {op}")

In [ ]:
# --- M2 threshold config ---
th_config = load_threshold_config(PATHS)
print("M2 Threshold Config:")
for sc in SCENARIOS:
    op = get_operating_point(th_config, sc)
    print(f"  {sc}: S1_th={op['s1_threshold']:.2f}, "
          f"S2_LOW={op['s2_zone_low']:.2f}, S2_HIGH={op['s2_zone_high']:.2f}")

# --- Add prediction columns ---
def add_predictions(df, stage, scenario, config):
    """Add prediction and confidence columns."""
    df = df.copy()
    op = get_operating_point(config, scenario)

    if stage == 1:
        prob_col = 'prob_IAS_cal'
        threshold = op['s1_threshold']
        df['y_true'] = df['target']
        df['y_pred'] = (df[prob_col] >= threshold).astype(int)
        df['y_true_label'] = df['y_true'].map(S1_LABELS)
        df['y_pred_label'] = df['y_pred'].map(S1_LABELS)
        df['confidence'] = df[prob_col].where(df['y_pred'] == 1,
                                               1 - df[prob_col])
        df['correct'] = (df['y_true'] == df['y_pred']).astype(int)
    else:
        prob_cols = ['prob_DEA_cal', 'prob_HA_cal',
                     'prob_HGB_HTZ_cal', 'prob_Normal_cal']
        probs = df[prob_cols].values
        df['y_true'] = df['target']
        df['y_pred'] = probs.argmax(axis=1)
        df['confidence'] = probs.max(axis=1)
        df['y_true_label'] = df['y_true'].map(S2_LABELS)
        df['y_pred_label'] = df['y_pred'].map(S2_LABELS)
        df['correct'] = (df['y_true'] == df['y_pred']).astype(int)

        # Assign zone
        low_th = op['s2_zone_low']
        high_th = op['s2_zone_high']
        df['zone'] = np.where(
            df['confidence'] >= high_th, 'HIGH',
            np.where(df['confidence'] < low_th, 'LOW', 'MEDIUM'))

    return df

# Apply to all data
for scenario in SCENARIOS:
    for stage in [1, 2]:
        for split in ['oof', 'test']:
            data[scenario][stage][split] = add_predictions(
                data[scenario][stage][split], stage, scenario, th_config)

# Check
for scenario in SCENARIOS:
    for split in ['oof', 'test']:
        df2 = data[scenario][2][split]
        acc = accuracy_score(df2['y_true'], df2['y_pred'])
        print(f"{scenario} S2 {split}: acc={acc:.3f}, "
              f"n={len(df2)}, correct={df2['correct'].sum()}")

print("\n✅ Predictions and zones added")

## 2. 3A — Confusion Matrix & Per-Class Metrics

### 2.1 Normalized Confusion Matrix — 2×2 Panel (S1+S2 × CBC_Only+CBC_BIO)

In [ ]:
def plot_confusion_matrix_panel(data, split='test', figsize=(14, 10)):
    """2×2 panel: (S1, S2) × (CBC_Only, CBC_BIO) normalized confusion matrix."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    # Tufte-compatible: gradient from white to PALETTE highlight
    from matplotlib.colors import LinearSegmentedColormap

    cmap_tufte = LinearSegmentedColormap.from_list(
        'tufte_cm', ['#FFFFFF', PALETTE['base1']], N=256)

    configs = [
        (0, 0, 'CBC_Only', 1, S1_DISPLAY, 'CBC_Only — Stage 1 (Binary)'),
        (0, 1, 'CBC_BIO',  1, S1_DISPLAY, 'CBC_BIO — Stage 1 (Binary)'),
        (1, 0, 'CBC_Only', 2, S2_DISPLAY, 'CBC_Only — Stage 2 (4-class)'),
        (1, 1, 'CBC_BIO',  2, S2_DISPLAY, 'CBC_BIO — Stage 2 (4-class)'),
    ]

    for row, col, scenario, stage, labels, title in configs:
        ax = axes[row, col]
        df = data[scenario][stage][split]
        cm = confusion_matrix(df['y_true'], df['y_pred'])
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_norm, interpolation='nearest', cmap=cmap_tufte,
                       vmin=0, vmax=1)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_ylabel('Actual', fontsize=9)
        ax.set_xlabel('Predicted', fontsize=9)
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
        ax.set_yticklabels(labels, fontsize=8)

        for i in range(len(labels)):
            for j in range(len(labels)):
                val = cm_norm[i, j]
                count = cm[i, j]
                color = 'white' if val > 0.5 else 'black'
                ax.text(j, i, f"{val:.2f}\n(n={count})",
                        ha='center', va='center', fontsize=8, color=color)

        despine_all(ax)

    fig.suptitle(f'Normalized Confusion Matrix — {split.upper()} Set',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_cm_oof = plot_confusion_matrix_panel(data, split='oof')
save_fig(fig_cm_oof, plots_dir, 'm3_confusion_matrix_oof')
plt.show()

In [ ]:
# --- TEST ---
fig_cm_test = plot_confusion_matrix_panel(data, split='test')
save_fig(fig_cm_test, plots_dir, 'm3_confusion_matrix_test')
plt.show()


### 2.2 Per-Class Metrics Table

In [ ]:
def compute_per_class_metrics(data, split='test'):
    """Per-class metrics for all scenarios and stages."""
    rows = []
    for scenario in SCENARIOS:
        for stage in [1, 2]:
            df = data[scenario][stage][split]
            labels = S1_NAMES if stage == 1 else S2_NAMES
            label_ids = list(range(len(labels)))

            prec, rec, f1, sup = precision_recall_fscore_support(
                df['y_true'], df['y_pred'], labels=label_ids,
                average=None, zero_division=0)

            for i, cls_name in enumerate(labels):
                rows.append({
                    'Scenario': scenario,
                    'Stage': f'Stage {stage}',
                    'Class': DISP_MAP.get(cls_name, cls_name),
                    'Precision': round(prec[i], 3),
                    'Recall': round(rec[i], 3),
                    'F1': round(f1[i], 3),
                    'Support': int(sup[i])
                })

            # Macro average
            rows.append({
                'Scenario': scenario,
                'Stage': f'Stage {stage}',
                'Class': 'MACRO AVG',
                'Precision': round(np.mean(prec), 3),
                'Recall': round(np.mean(rec), 3),
                'F1': round(np.mean(f1), 3),
                'Support': int(np.sum(sup))
            })

    return pd.DataFrame(rows)

# OOF & Test
metrics_oof = compute_per_class_metrics(data, 'oof')
metrics_test = compute_per_class_metrics(data, 'test')

# Excel kaydet
with pd.ExcelWriter(os.path.join(analysis_dir, 'per_class_metrics.xlsx')) as w:
    metrics_oof.to_excel(w, sheet_name='OOF', index=False)
    metrics_test.to_excel(w, sheet_name='Test', index=False)

print("=== TEST Per-Class Metrics ===")
display(metrics_test)
print(f"\n💾 per_class_metrics.xlsx saved → {analysis_dir}")

### 2.3 CBC_Only vs CBC_BIO Improvement Table

In [ ]:
def scenario_improvement_table(metrics_df):
    """Improvement table of CBC_BIO relative to CBC_Only."""
    rows = []
    for stage in ['Stage 1', 'Stage 2']:
        df_stage = metrics_df[metrics_df['Stage'] == stage]
        cbc = df_stage[df_stage['Scenario'] == 'CBC_Only'].set_index('Class')
        bio = df_stage[df_stage['Scenario'] == 'CBC_BIO'].set_index('Class')

        common_classes = cbc.index.intersection(bio.index)
        for cls in common_classes:
            for metric in ['Precision', 'Recall', 'F1']:
                val_cbc = cbc.loc[cls, metric]
                val_bio = bio.loc[cls, metric]
                delta = val_bio - val_cbc
                rows.append({
                    'Stage': stage, 'Class': cls, 'Metric': metric,
                    'CBC_Only': val_cbc, 'CBC_BIO': val_bio,
                    'Delta': round(delta, 3),
                    'Improvement': f"{delta:+.3f}" if delta != 0 else "—"
                })
    return pd.DataFrame(rows)

improvement_df = scenario_improvement_table(metrics_test)
improvement_df.to_excel(os.path.join(analysis_dir, 'scenario_improvement.xlsx'),
                        index=False)
display(improvement_df)
print(f"\n💾 scenario_improvement.xlsx saved")

### 2.4 Zone-Stratified Confusion Matrix (Stage 2 — HIGH vs MEDIUM)

In [ ]:
def plot_zone_confusion_matrices(data, split='test', figsize=(16, 6)):
    """Separate confusion matrices for HIGH vs MEDIUM zones — 2×2 panel."""
    fig, axes = plt.subplots(1, 4, figsize=figsize)

    configs = [
        (0, 'CBC_Only', 'HIGH',   'CBC_Only — HIGH Zone'),
        (1, 'CBC_Only', 'MEDIUM', 'CBC_Only — MEDIUM Zone'),
        (2, 'CBC_BIO',  'HIGH',   'CBC_BIO — HIGH Zone'),
        (3, 'CBC_BIO',  'MEDIUM', 'CBC_BIO — MEDIUM Zone'),
    ]

    for idx, scenario, zone, title in configs:
        ax = axes[idx]
        df = data[scenario][2][split]
        df_zone = df[df['zone'] == zone]

        if len(df_zone) == 0:
            ax.text(0.5, 0.5, f'n=0\n(zone empty)', ha='center',
                    va='center', fontsize=12, transform=ax.transAxes)
            ax.set_title(title, fontsize=9, fontweight='bold')
            despine_all(ax)
            continue

        cm = confusion_matrix(df_zone['y_true'], df_zone['y_pred'],
                              labels=[0, 1, 2, 3])
        row_sums = cm.sum(axis=1, keepdims=True)
        # Zero-division guard
        cm_norm = np.divide(cm.astype(float), row_sums,
                           out=np.zeros_like(cm, dtype=float),
                           where=row_sums != 0)

        im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues',
                       vmin=0, vmax=1)
        ax.set_title(f"{title}\n(n={len(df_zone)}, "
                     f"acc={accuracy_score(df_zone['y_true'], df_zone['y_pred']):.2f})",
                     fontsize=9, fontweight='bold')
        ax.set_xticks(range(4))
        ax.set_yticks(range(4))
        ax.set_xticklabels(S2_DISPLAY, fontsize=7, rotation=45, ha='right')
        ax.set_yticklabels(S2_DISPLAY, fontsize=7)
        ax.set_ylabel('Actual', fontsize=8)
        ax.set_xlabel('Predicted', fontsize=8)

        for i in range(4):
            for j in range(4):
                val = cm_norm[i, j]
                count = cm[i, j]
                if count > 0:
                    color = 'white' if val > 0.5 else 'black'
                    ax.text(j, i, f"{val:.2f}\n({count})",
                            ha='center', va='center', fontsize=7, color=color)

        despine_all(ax)

    fig.suptitle(f'Zone-Stratified Confusion Matrix — Stage 2 {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_zone_cm = plot_zone_confusion_matrices(data, 'test')
save_fig(fig_zone_cm, plots_dir, 'm3_zone_confusion_matrix_test')
plt.show()

### 2.5 Save Confusion Matrices to Excel

In [ ]:
# Collect all CMs
with pd.ExcelWriter(os.path.join(analysis_dir, 'confusion_matrices.xlsx')) as w:
    for scenario in SCENARIOS:
        for stage in [1, 2]:
            labels = S1_DISPLAY if stage == 1 else S2_DISPLAY
            for split in ['oof', 'test']:
                df = data[scenario][stage][split]
                cm = confusion_matrix(df['y_true'], df['y_pred'])
                cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

                # Raw
                df_raw = pd.DataFrame(cm, index=labels, columns=labels)
                df_raw.to_excel(w, sheet_name=f"{scenario}_S{stage}_{split}_raw")

                # Normalized
                df_norm = pd.DataFrame(np.round(cm_norm, 3),
                                       index=labels, columns=labels)
                df_norm.to_excel(w, sheet_name=f"{scenario}_S{stage}_{split}_norm")

    # Zone-stratified (Stage 2 test)
    for scenario in SCENARIOS:
        df = data[scenario][2]['test']
        for zone in ['HIGH', 'MEDIUM', 'LOW']:
            df_z = df[df['zone'] == zone]
            if len(df_z) > 0:
                cm = confusion_matrix(df_z['y_true'], df_z['y_pred'],
                                      labels=[0,1,2,3])
                cm_df = pd.DataFrame(cm, index=S2_DISPLAY, columns=S2_DISPLAY)
                cm_df.to_excel(w, sheet_name=f"{scenario}_{zone}_test")

print(f"💾 confusion_matrices.xlsx saved → {analysis_dir}")

## 3. 3B — Misclassification Pattern Analysis

### 3.1 Most Frequently Confused Class Pairs — Error Flow Heatmap

In [ ]:
def misclassification_flow_heatmap(data, split='test', figsize=(14, 5)):
    """Misclassification flow heatmap per scenario (off-diagonal only)."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for idx, scenario in enumerate(SCENARIOS):
        ax = axes[idx]
        df = data[scenario][2][split]
        df_err = df[df['correct'] == 0]

        # Error-pair frequencies
        cm = confusion_matrix(df['y_true'], df['y_pred'], labels=[0,1,2,3])
        # Off-diagonal: errors only
        cm_err = cm.copy()
        np.fill_diagonal(cm_err, 0)

        im = ax.imshow(cm_err, cmap='Reds', interpolation='nearest')
        ax.set_title(f'{scenario} — Misclassification Flow\n'
                     f'(n_errors={len(df_err)}/{len(df)})',
                     fontsize=10, fontweight='bold')
        ax.set_xticks(range(4))
        ax.set_yticks(range(4))
        ax.set_xticklabels(S2_DISPLAY, fontsize=8, rotation=45, ha='right')
        ax.set_yticklabels(S2_DISPLAY, fontsize=8)
        ax.set_ylabel('Actual Class', fontsize=9)
        ax.set_xlabel('Misclassified As', fontsize=9)

        for i in range(4):
            for j in range(4):
                val = cm_err[i, j]
                if val > 0:
                    ax.text(j, i, str(val), ha='center', va='center',
                            fontsize=10, fontweight='bold',
                            color='white' if val > cm_err.max() * 0.5 else 'black')

        despine_all(ax)

    fig.suptitle(f'Misclassification Flow — Stage 2 {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_flow = misclassification_flow_heatmap(data, 'test')
save_fig(fig_flow, plots_dir, 'm3_misclassification_flow_test')
plt.show()

### 3.2 Top Error Pairs Table

In [ ]:
def top_error_pairs(data, split='test', top_n=10):
    """Most frequently confused class pairs."""
    all_rows = []
    for scenario in SCENARIOS:
        df = data[scenario][2][split]
        df_err = df[df['correct'] == 0]

        pair_counts = df_err.groupby(['y_true_label', 'y_pred_label']).agg(
            count=('correct', 'size'),
            mean_confidence=('confidence', 'mean'),
            median_confidence=('confidence', 'median'),
            min_confidence=('confidence', 'min'),
            max_confidence=('confidence', 'max'),
        ).reset_index()

        pair_counts['Scenario'] = scenario
        pair_counts = pair_counts.sort_values('count', ascending=False)
        all_rows.append(pair_counts)

    result = pd.concat(all_rows, ignore_index=True)
    result.columns = ['True_Class', 'Pred_Class', 'Error_Count',
                      'Mean_Conf', 'Median_Conf', 'Min_Conf', 'Max_Conf',
                      'Scenario']
    # Compute percentage
    for scenario in SCENARIOS:
        mask = result['Scenario'] == scenario
        total = data[scenario][2][split].shape[0]
        result.loc[mask, 'Error_Pct'] = (
            result.loc[mask, 'Error_Count'] / total * 100).round(1)

    # Display labels for publication (internal y_*_label preserved upstream)
    result['True_Class'] = result['True_Class'].map(lambda c: DISP_MAP.get(c, c))
    result['Pred_Class'] = result['Pred_Class'].map(lambda c: DISP_MAP.get(c, c))

    return result[['Scenario', 'True_Class', 'Pred_Class', 'Error_Count',
                    'Error_Pct', 'Mean_Conf', 'Median_Conf',
                    'Min_Conf', 'Max_Conf']]

error_pairs_df = top_error_pairs(data, 'test')
error_pairs_df.to_excel(
    os.path.join(analysis_dir, 'misclassification_patterns.xlsx'), index=False)

display(error_pairs_df)
print(f"\n💾 misclassification_patterns.xlsx saved")

### 3.3 Confidence Distribution — Correct vs Incorrect

In [ ]:
def plot_confidence_distribution(data, split='test', figsize=(14, 10)):
    """Confidence distribution of correct vs incorrect predictions — violin + strip."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    configs = [
        (0, 0, 'CBC_Only', 1, 'CBC_Only — Stage 1'),
        (0, 1, 'CBC_BIO',  1, 'CBC_BIO — Stage 1'),
        (1, 0, 'CBC_Only', 2, 'CBC_Only — Stage 2'),
        (1, 1, 'CBC_BIO',  2, 'CBC_BIO — Stage 2'),
    ]

    for row, col, scenario, stage, title in configs:
        ax = axes[row, col]
        df = data[scenario][stage][split].copy()
        df['Status'] = df['correct'].map({1: 'Correct', 0: 'Incorrect'})

        colors = [PALETTE['accent1'], PALETTE['highlight']]
        sns.violinplot(data=df, x='Status', y='confidence', ax=ax,
                       palette=colors, inner='quartile', cut=0, alpha=0.7)
        sns.stripplot(data=df, x='Status', y='confidence', ax=ax,
                      color='black', alpha=0.15, size=2, jitter=True)

        # Statistics
        for i, status in enumerate(['Correct', 'Incorrect']):
            sub = df[df['Status'] == status]['confidence']
            if len(sub) > 0:
                ax.text(i, sub.median() + 0.02,
                        f"med={sub.median():.2f}\nn={len(sub)}",
                        ha='center', fontsize=7, fontweight='bold')

        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_ylabel('Confidence', fontsize=9)
        ax.set_xlabel('')
        ax.set_ylim(0, 1.05)
        despine_all(ax)

    fig.suptitle(f'Confidence Distribution: Correct vs Incorrect — {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_conf = plot_confidence_distribution(data, 'test')
save_fig(fig_conf, plots_dir, 'm3_confidence_distribution_test')
plt.show()

### 3.4 Zone-Based Error Analysis — MEDIUM Zone Detail

In [ ]:
def zone_error_analysis(data, split='test'):
    """Detailed per-zone error statistics."""
    rows = []
    for scenario in SCENARIOS:
        df = data[scenario][2][split]
        for zone in ['HIGH', 'MEDIUM', 'LOW']:
            df_z = df[df['zone'] == zone]
            if len(df_z) == 0:
                continue
            n_total = len(df_z)
            n_correct = df_z['correct'].sum()
            n_error = n_total - n_correct

            # Class distribution
            class_dist = {DISP_MAP.get(k, k): v
                          for k, v in df_z['y_true_label'].value_counts().to_dict().items()}

            # Most frequent error pairs
            df_err = df_z[df_z['correct'] == 0]
            if len(df_err) > 0:
                top_pair = df_err.groupby(
                    ['y_true_label', 'y_pred_label']).size().idxmax()
                top_pair_count = df_err.groupby(
                    ['y_true_label', 'y_pred_label']).size().max()
                top_pair_str = f"{DISP_MAP.get(top_pair[0], top_pair[0])}→{DISP_MAP.get(top_pair[1], top_pair[1])} (n={top_pair_count})"
            else:
                top_pair_str = "—"

            rows.append({
                'Scenario': scenario,
                'Zone': zone,
                'N': n_total,
                'Pct_of_total': round(n_total / len(df) * 100, 1),
                'Accuracy': round(n_correct / n_total, 3),
                'N_errors': n_error,
                'Mean_conf': round(df_z['confidence'].mean(), 3),
                'Top_error_pair': top_pair_str,
                'Class_dist': str(class_dist),
            })

    return pd.DataFrame(rows)

zone_analysis = zone_error_analysis(data, 'test')
display(zone_analysis)

# MEDIUM zone detail
print("\n=== MEDIUM Zone Detailed Error Breakdown ===")
for scenario in SCENARIOS:
    df = data[scenario][2]['test']
    df_med = df[df['zone'] == 'MEDIUM']
    if len(df_med) == 0:
        print(f"  {scenario}: MEDIUM zone empty")
        continue
    df_err = df_med[df_med['correct'] == 0]
    print(f"\n  {scenario} MEDIUM: n={len(df_med)}, "
          f"errors={len(df_err)} ({len(df_err)/len(df_med)*100:.1f}%)")
    if len(df_err) > 0:
        pairs = df_err.groupby(['y_true_label', 'y_pred_label']).size()
        pairs = pairs.sort_values(ascending=False)
        print(f"  Error pairs:")
        for (true_cls, pred_cls), count in pairs.items():
            print(f"    {DISP_MAP.get(true_cls, true_cls)} → {DISP_MAP.get(pred_cls, pred_cls)}: {count}")

### 3.5 Per-Class Confidence by Correctness — Stage 2

In [ ]:
def plot_per_class_confidence(data, split='test', figsize=(14, 5)):
    """Per-class confidence distribution (correct vs incorrect)."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for idx, scenario in enumerate(SCENARIOS):
        ax = axes[idx]
        df = data[scenario][2][split].copy()
        df['Status'] = df['correct'].map({1: 'Correct', 0: 'Incorrect'})

        # Box plot per class
        positions = []
        labels_x = []
        bp_data_correct = []
        bp_data_incorrect = []

        for i, cls in enumerate(S2_NAMES):
            correct_vals = df[(df['y_true_label'] == cls) &
                             (df['Status'] == 'Correct')]['confidence']
            incorrect_vals = df[(df['y_true_label'] == cls) &
                               (df['Status'] == 'Incorrect')]['confidence']
            bp_data_correct.append(correct_vals.values if len(correct_vals) > 0
                                   else [np.nan])
            bp_data_incorrect.append(incorrect_vals.values if len(incorrect_vals) > 0
                                     else [np.nan])

        x = np.arange(len(S2_NAMES))
        width = 0.35
        bp1 = ax.boxplot(bp_data_correct, positions=x - width/2,
                         widths=width*0.8, patch_artist=True,
                         boxprops=dict(facecolor=PALETTE['accent1'], alpha=0.6),
                         medianprops=dict(color='black'))
        bp2 = ax.boxplot(bp_data_incorrect, positions=x + width/2,
                         widths=width*0.8, patch_artist=True,
                         boxprops=dict(facecolor=PALETTE['highlight'], alpha=0.6),
                         medianprops=dict(color='black'))

        ax.set_xticks(x)
        ax.set_xticklabels(S2_DISPLAY, fontsize=8)
        ax.set_ylabel('Confidence', fontsize=9)
        ax.set_title(f'{scenario} — Per-Class Confidence', fontsize=10,
                     fontweight='bold')
        ax.legend([bp1['boxes'][0], bp2['boxes'][0]],
                  ['Correct', 'Incorrect'], fontsize=8, loc='lower left')
        ax.set_ylim(0, 1.05)
        despine_all(ax)

    fig.suptitle(f'Per-Class Confidence — Stage 2 {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_cls_conf = plot_per_class_confidence(data, 'test')
save_fig(fig_cls_conf, plots_dir, 'm3_per_class_confidence_test')
plt.show()

## 4. 3C — Clinical Error Impact Analysis

### 4.1 Defining the Clinical Risk Matrix

In [ ]:
# --- Clinical error severity scores ---
# 1 = low risk (treatment unchanged), 2 = moderate risk, 3 = high risk (treatment misdirected)

CLINICAL_SEVERITY = {
    # (true_class, pred_class) : (severity, explanation)
    # Stage 2 errors
    ('DEA', 'HA'):        (2, 'Iron therapy omitted; hemolytic anemia workup initiated'),
    ('DEA', 'HGB_HTZ'):   (3, 'Iron therapy omitted; referral for genetic testing'),
    ('DEA', 'Normal'):    (3, 'IDA missed — delayed initiation of iron therapy'),
    ('HA', 'DEA'):        (2, 'Unnecessary iron therapy; hemolysis workup delayed'),
    ('HA', 'HGB_HTZ'):    (2, 'Hemolysis treatment omitted; referral for genetic testing'),
    ('HA', 'Normal'):     (3, 'Hemolysis missed — potential crisis risk'),
    ('HGB_HTZ', 'DEA'):   (3, 'Genetic counseling omitted; unnecessary iron therapy initiated'),
    ('HGB_HTZ', 'HA'):    (2, 'Genetic counseling omitted; hemolysis workup initiated'),
    ('HGB_HTZ', 'Normal'): (2, 'Carrier status overlooked — family planning affected'),
    ('Normal', 'DEA'):    (2, 'Unnecessary iron therapy initiated'),
    ('Normal', 'HA'):     (2, 'Unnecessary hemolysis workup'),
    ('Normal', 'HGB_HTZ'): (1, 'Unnecessary genetic testing — low clinical risk'),

    # Stage 1 errors
    ('DAS', 'IAS'):  (1, 'Unnecessary further testing — low risk but increased cost'),
    ('IAS', 'DAS'):  (3, 'Anemia-associated disease missed — treatment delay'),
}

# Color mapping
SEVERITY_COLORS = {1: '#27AE60', 2: '#E67E22', 3: '#C0392B'}
SEVERITY_LABELS = {1: 'Low', 2: 'Medium', 3: 'High'}

print("✅ Clinical severity matrix defined")
print(f"   {len(CLINICAL_SEVERITY)} error types categorized")

### 4.2 Clinical Risk Matrix — Combining with Error Counts

In [ ]:
def build_clinical_risk_table(data, split='test'):
    """Combine error counts with clinical severity scores."""
    rows = []
    for scenario in SCENARIOS:
        # Stage 1
        df1 = data[scenario][1][split]
        df1_err = df1[df1['correct'] == 0]
        for (true_cls, pred_cls), (sev, desc) in CLINICAL_SEVERITY.items():
            if true_cls in S1_NAMES and pred_cls in S1_NAMES:
                count = len(df1_err[(df1_err['y_true_label'] == true_cls) &
                                    (df1_err['y_pred_label'] == pred_cls)])
                if count > 0:
                    rows.append({
                        'Scenario': scenario, 'Stage': 'Stage 1',
                        'Actual': DISP_MAP.get(true_cls, true_cls),
                        'Predicted': DISP_MAP.get(pred_cls, pred_cls),
                        'Error_Count': count,
                        'Severity': sev, 'Severity_Label': SEVERITY_LABELS[sev],
                        'Clinical_Impact': desc,
                        'Weighted_Risk': count * sev
                    })

        # Stage 2
        df2 = data[scenario][2][split]
        df2_err = df2[df2['correct'] == 0]
        for (true_cls, pred_cls), (sev, desc) in CLINICAL_SEVERITY.items():
            if true_cls in S2_NAMES and pred_cls in S2_NAMES:
                count = len(df2_err[(df2_err['y_true_label'] == true_cls) &
                                    (df2_err['y_pred_label'] == pred_cls)])
                if count > 0:
                    rows.append({
                        'Scenario': scenario, 'Stage': 'Stage 2',
                        'Actual': DISP_MAP.get(true_cls, true_cls),
                        'Predicted': DISP_MAP.get(pred_cls, pred_cls),
                        'Error_Count': count,
                        'Severity': sev, 'Severity_Label': SEVERITY_LABELS[sev],
                        'Clinical_Impact': desc,
                        'Weighted_Risk': count * sev
                    })

    result = pd.DataFrame(rows)
    result = result.sort_values(['Scenario', 'Stage', 'Weighted_Risk'],
                                ascending=[True, True, False])
    return result

risk_table = build_clinical_risk_table(data, 'test')
risk_table.to_excel(os.path.join(analysis_dir, 'clinical_risk_matrix.xlsx'),
                    index=False)
display(risk_table)
print(f"\n💾 clinical_risk_matrix.xlsx saved")

### 4.3 Clinical Risk Heatmap

In [ ]:
def plot_clinical_risk_heatmap(data, split='test', figsize=(14, 5)):
    """Severity-weighted error heatmap — Stage 2."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for idx, scenario in enumerate(SCENARIOS):
        ax = axes[idx]
        df = data[scenario][2][split]

        cm = confusion_matrix(df['y_true'], df['y_pred'], labels=[0,1,2,3])
        # Build the severity matrix
        sev_matrix = np.zeros((4, 4))
        for i, true_cls in enumerate(S2_NAMES):
            for j, pred_cls in enumerate(S2_NAMES):
                if i != j:  # off-diagonal
                    key = (true_cls, pred_cls)
                    sev = CLINICAL_SEVERITY.get(key, (0, ''))[0]
                    sev_matrix[i, j] = cm[i, j] * sev  # weighted risk

        im = ax.imshow(sev_matrix, cmap='YlOrRd', interpolation='nearest')
        ax.set_title(f'{scenario} — Weighted Clinical Risk\n'
                     f'(Count × Severity)',
                     fontsize=10, fontweight='bold')
        ax.set_xticks(range(4))
        ax.set_yticks(range(4))
        ax.set_xticklabels(S2_DISPLAY, fontsize=8, rotation=45, ha='right')
        ax.set_yticklabels(S2_DISPLAY, fontsize=8)
        ax.set_ylabel('Actual', fontsize=9)
        ax.set_xlabel('Predicted', fontsize=9)

        for i in range(4):
            for j in range(4):
                if i == j:
                    ax.text(j, i, '—', ha='center', va='center',
                            fontsize=9, color='gray')
                else:
                    val = sev_matrix[i, j]
                    count = cm[i, j]
                    sev = CLINICAL_SEVERITY.get(
                        (S2_NAMES[i], S2_NAMES[j]), (0, ''))[0]
                    if count > 0:
                        color = 'white' if val > sev_matrix.max() * 0.5 else 'black'
                        ax.text(j, i, f"{int(count)}\n(sev={sev})",
                                ha='center', va='center', fontsize=8,
                                color=color, fontweight='bold')

        despine_all(ax)

    fig.suptitle(f'Clinical Risk Heatmap — Stage 2 {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_risk = plot_clinical_risk_heatmap(data, 'test')
save_fig(fig_risk, plots_dir, 'm3_clinical_risk_heatmap_test')
plt.show()

### 4.4 Safe vs Dangerous Error Summary

In [ ]:
def safe_vs_dangerous_summary(risk_table):
    """Error summary by severity level."""
    summary_rows = []
    for scenario in SCENARIOS:
        rt = risk_table[risk_table['Scenario'] == scenario]
        for sev in [1, 2, 3]:
            sub = rt[rt['Severity'] == sev]
            summary_rows.append({
                'Scenario': scenario,
                'Severity': SEVERITY_LABELS[sev],
                'N_error_types': len(sub),
                'Total_errors': sub['Error_Count'].sum(),
                'Total_weighted_risk': sub['Weighted_Risk'].sum(),
            })
    summary = pd.DataFrame(summary_rows)

    # Total error percentage
    for scenario in SCENARIOS:
        mask = summary['Scenario'] == scenario
        total = summary.loc[mask, 'Total_errors'].sum()
        summary.loc[mask, 'Pct_of_errors'] = (
            summary.loc[mask, 'Total_errors'] / total * 100).round(1)

    return summary

risk_summary = safe_vs_dangerous_summary(risk_table)
display(risk_summary)

print("\n📊 Interpretation:")
for scenario in SCENARIOS:
    rs = risk_summary[risk_summary['Scenario'] == scenario]
    high_risk = rs[rs['Severity'] == 'High']
    if len(high_risk) > 0:
        pct = high_risk['Pct_of_errors'].values[0]
        n = high_risk['Total_errors'].values[0]
        print(f"  {scenario}: Share of high-risk errors: {pct}% ({n} errors)")

## 5. 3D — Feature Contribution to Errors

### 5.1 Feature Profile Comparison in Correct vs Incorrect Predictions

In [ ]:
def get_feature_columns(df, scenario):
    """Extract feature columns from the data columns."""
    exclude_patterns = ['target', 'prob_', 'calibration_method',
                        'y_true', 'y_pred', 'confidence', 'correct',
                        'zone', 'y_true_label', 'y_pred_label',
                        'fold', 'Unnamed']
    exclude_exact = {'pred'}  # Stage 2 argmax prediction, not a feature
    feat_cols = [c for c in df.columns
                 if c not in exclude_exact
                 and not any(p in c for p in exclude_patterns)
                 and df[c].dtype in ['float64', 'float32', 'int64', 'int32']]
    return feat_cols

# Test
for scenario in SCENARIOS:
    df = data[scenario][2]['test']
    feat_cols = get_feature_columns(df, scenario)
    print(f"{scenario}: {len(feat_cols)} features → {feat_cols[:10]}...")

### 5.2 Feature Profile — Radar Chart (Top Features)

In [ ]:
def plot_feature_radar(data, scenario, split='test', top_n=8, figsize=(8, 8)):
    """Radar chart of top-N features for correct vs incorrect predictions."""
    df = data[scenario][2][split].copy()
    feat_cols = get_feature_columns(df, scenario)

    # Find the features that differ most between correct vs incorrect
    correct_means = df[df['correct'] == 1][feat_cols].mean()
    incorrect_means = df[df['correct'] == 0][feat_cols].mean()
    overall_std = df[feat_cols].std().replace(0, np.nan)

    # Cohen's d-like effect size
    effect_size = ((correct_means - incorrect_means) / overall_std).abs()
    effect_size = effect_size.dropna().sort_values(ascending=False)
    top_features = effect_size.head(top_n).index.tolist()

    if len(top_features) < 3:
        print(f"  {scenario}: Not enough features, skipping")
        return None

    # Z-score normalize
    df_z = df[top_features].copy()
    for col in top_features:
        mu, sigma = df[col].mean(), df[col].std()
        if sigma > 0:
            df_z[col] = (df[col] - mu) / sigma

    correct_z = df_z[df['correct'] == 1].mean()
    incorrect_z = df_z[df['correct'] == 0].mean()

    # Radar plot
    angles = np.linspace(0, 2 * np.pi, len(top_features), endpoint=False)
    angles = np.concatenate([angles, [angles[0]]])

    correct_vals = np.concatenate([correct_z.values, [correct_z.values[0]]])
    incorrect_vals = np.concatenate([incorrect_z.values, [incorrect_z.values[0]]])

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(projection='polar'))
    ax.fill(angles, correct_vals, alpha=0.15, color=PALETTE['accent1'])
    ax.plot(angles, correct_vals, 'o-', color=PALETTE['accent1'],
            linewidth=2, label='Correct', markersize=4)
    ax.fill(angles, incorrect_vals, alpha=0.15, color=PALETTE['highlight'])
    ax.plot(angles, incorrect_vals, 'o-', color=PALETTE['highlight'],
            linewidth=2, label='Incorrect', markersize=4)

    # Shortened feature names
    short_names = [feat_disp(f) for f in top_features]
    ax.set_xticklabels(short_names, fontsize=8)
    ax.set_title(f'{scenario} — Feature Profile\n(Z-scored, Top-{top_n} by effect size)',
                 fontsize=11, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    ax.set_ylim(min(min(correct_vals), min(incorrect_vals)) - 0.3,
                max(max(correct_vals), max(incorrect_vals)) + 0.3)

    return fig

fig_radar_cbc = plot_feature_radar(data, 'CBC_Only', 'test')
if fig_radar_cbc:
    save_fig(fig_radar_cbc, plots_dir, 'm3_feature_radar_cbc_only')
    plt.show()

In [ ]:
fig_radar_bio = plot_feature_radar(data, 'CBC_BIO', 'test')
if fig_radar_bio:
    save_fig(fig_radar_bio, plots_dir, 'm3_feature_radar_cbc_bio')
    plt.show()

### 5.3 Feature Distribution — Correct vs Incorrect (Box Plots)

In [ ]:
def plot_feature_comparison_boxes(data, scenario, split='test',
                                  top_n=6, figsize=(14, 8)):
    """Box-plot comparison of correct vs incorrect for top-N features."""
    df = data[scenario][2][split].copy()
    feat_cols = get_feature_columns(df, scenario)

    # Most differentiating features
    correct_means = df[df['correct'] == 1][feat_cols].mean()
    incorrect_means = df[df['correct'] == 0][feat_cols].mean()
    overall_std = df[feat_cols].std().replace(0, np.nan)
    effect_size = ((correct_means - incorrect_means) / overall_std).abs()
    effect_size = effect_size.dropna().sort_values(ascending=False)
    top_features = effect_size.head(top_n).index.tolist()

    n_feat = len(top_features)
    if n_feat == 0:
        return None

    ncols = 3
    nrows = (n_feat + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.atleast_2d(axes)

    df['Status'] = df['correct'].map({1: 'Correct', 0: 'Incorrect'})

    for i, feat in enumerate(top_features):
        row, col = divmod(i, ncols)
        ax = axes[row, col]
        colors = [PALETTE['accent1'], PALETTE['highlight']]
        sns.boxplot(data=df, x='Status', y=feat, ax=ax,
                    palette=colors, width=0.5)
        ax.set_title(f'{feat_disp(feat)}\n(d={effect_size[feat]:.2f})',
                     fontsize=9, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        despine_all(ax)

    # Hide empty subplots
    for i in range(n_feat, nrows * ncols):
        row, col = divmod(i, ncols)
        axes[row, col].set_visible(False)

    fig.suptitle(f'{scenario} — Feature Comparison (Correct vs Incorrect)\n'
                 f'Stage 2 {split.upper()}',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_box_cbc = plot_feature_comparison_boxes(data, 'CBC_Only', 'test')
if fig_box_cbc:
    save_fig(fig_box_cbc, plots_dir, 'm3_feature_boxes_cbc_only')
    plt.show()

In [ ]:
fig_box_bio = plot_feature_comparison_boxes(data, 'CBC_BIO', 'test')
if fig_box_bio:
    save_fig(fig_box_bio, plots_dir, 'm3_feature_boxes_cbc_bio')
    plt.show()

## 6. `src/m3_errors.py` — Module Creation

Functions usable in later chats via `from m3_errors import ...`.

In [ ]:
m3_module_code = '''"""
M3 Error Analysis -- Error analysis functions.
CDS Pipeline Module 3.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             accuracy_score)

# --- Constants ---
S1_LABELS = {0: 'DAS', 1: 'IAS'}
S2_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}
S1_NAMES  = ['DAS', 'IAS']
S2_NAMES  = ['DEA', 'HA', 'HGB_HTZ', 'Normal']

CLINICAL_SEVERITY = {
    ('DEA', 'HA'):       (2, 'Iron therapy omitted; hemolytic anemia workup initiated'),
    ('DEA', 'HGB_HTZ'):  (3, 'Iron therapy omitted; referral for genetic testing'),
    ('DEA', 'Normal'):   (3, 'IDA missed — delayed initiation of iron therapy'),
    ('HA', 'DEA'):       (2, 'Unnecessary iron therapy; hemolysis workup delayed'),
    ('HA', 'HGB_HTZ'):   (2, 'Hemolysis treatment omitted; referral for genetic testing'),
    ('HA', 'Normal'):    (3, 'Hemolysis missed — potential crisis risk'),
    ('HGB_HTZ', 'DEA'):  (3, 'Genetic counseling omitted; unnecessary iron therapy initiated'),
    ('HGB_HTZ', 'HA'):   (2, 'Genetic counseling omitted; hemolysis workup initiated'),
    ('HGB_HTZ', 'Normal'): (2, 'Carrier status overlooked — family planning affected'),
    ('Normal', 'DEA'):   (2, 'Unnecessary iron therapy initiated'),
    ('Normal', 'HA'):    (2, 'Unnecessary hemolysis workup'),
    ('Normal', 'HGB_HTZ'): (1, 'Unnecessary genetic testing — low clinical risk'),
    ('DAS', 'IAS'):  (1, 'Unnecessary further testing — low risk but increased cost'),
    ('IAS', 'DAS'):  (3, 'Anemia-associated disease missed — treatment delay'),
}

SEVERITY_LABELS = {1: 'Low', 2: 'Medium', 3: 'High'}


def get_misclassified(df, stage=2):
    """Return misclassified samples."""
    return df[df['correct'] == 0].copy()


def confusion_summary(df, stage=2):
    """Return confusion matrix + normalized version."""
    labels = S1_NAMES if stage == 1 else S2_NAMES
    label_ids = list(range(len(labels)))
    cm = confusion_matrix(df['y_true'], df['y_pred'], labels=label_ids)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    return {
        'raw': pd.DataFrame(cm, index=labels, columns=labels),
        'normalized': pd.DataFrame(np.round(cm_norm, 3),
                                   index=labels, columns=labels)
    }


def per_class_metrics(df, stage=2):
    """Compute per-class precision, recall, F1."""
    labels = S1_NAMES if stage == 1 else S2_NAMES
    label_ids = list(range(len(labels)))
    prec, rec, f1, sup = precision_recall_fscore_support(
        df['y_true'], df['y_pred'], labels=label_ids,
        average=None, zero_division=0)
    return pd.DataFrame({
        'Class': labels,
        'Precision': np.round(prec, 3),
        'Recall': np.round(rec, 3),
        'F1': np.round(f1, 3),
        'Support': sup.astype(int)
    })


def clinical_risk_score(df, stage=2):
    """Compute clinical risk scores for errors."""
    df_err = df[df['correct'] == 0].copy()
    if len(df_err) == 0:
        return pd.DataFrame()

    labels = S1_NAMES if stage == 1 else S2_NAMES
    rows = []
    for (true_cls, pred_cls), (sev, desc) in CLINICAL_SEVERITY.items():
        if true_cls in labels and pred_cls in labels:
            count = len(df_err[(df_err['y_true_label'] == true_cls) &
                               (df_err['y_pred_label'] == pred_cls)])
            if count > 0:
                rows.append({
                    'Actual': true_cls, 'Predicted': pred_cls,
                    'Error_Count': count, 'Severity': sev,
                    'Severity_Label': SEVERITY_LABELS[sev],
                    'Clinical_Impact': desc,
                    'Weighted_Risk': count * sev
                })

    return pd.DataFrame(rows).sort_values('Weighted_Risk', ascending=False)


def error_pair_frequencies(df, stage=2):
    """Error pair frequencies + confidence statistics."""
    df_err = df[df['correct'] == 0]
    if len(df_err) == 0:
        return pd.DataFrame()

    pairs = df_err.groupby(['y_true_label', 'y_pred_label']).agg(
        count=('correct', 'size'),
        mean_confidence=('confidence', 'mean'),
        median_confidence=('confidence', 'median'),
    ).reset_index()
    pairs.columns = ['True_Class', 'Pred_Class', 'Count',
                     'Mean_Conf', 'Median_Conf']
    return pairs.sort_values('Count', ascending=False)
'''

# Write to file
m3_path = os.path.join(PATHS.src, 'm3_errors.py')
with open(m3_path, 'w', encoding='utf-8') as f:
    f.write(m3_module_code)

print(f"m3_errors.py saved -> {m3_path}")

# Import test
spec = importlib.util.spec_from_file_location("m3_errors", m3_path)
m3_mod = importlib.util.module_from_spec(spec)
sys.modules['m3_errors'] = m3_mod
spec.loader.exec_module(m3_mod)

from m3_errors import get_misclassified, confusion_summary, clinical_risk_score
print("Import test passed")

## 7. OOF-Test Consistency Check

Verify that the error patterns observed in OOF also hold in the test set.

In [ ]:
def oof_test_consistency(data):
    """OOF vs Test per-class recall comparison."""
    rows = []
    for scenario in SCENARIOS:
        for stage in [1, 2]:
            labels = S1_NAMES if stage == 1 else S2_NAMES
            label_ids = list(range(len(labels)))

            for split in ['oof', 'test']:
                df = data[scenario][stage][split]
                _, rec, _, sup = precision_recall_fscore_support(
                    df['y_true'], df['y_pred'], labels=label_ids,
                    average=None, zero_division=0)
                for i, cls in enumerate(labels):
                    rows.append({
                        'Scenario': scenario, 'Stage': f'S{stage}',
                        'Class': DISP_MAP.get(cls, cls), 'Split': split,
                        'Recall': round(rec[i], 3), 'Support': int(sup[i])
                    })

    df_all = pd.DataFrame(rows)

    # OOF vs Test recall difference
    pivot = df_all.pivot_table(index=['Scenario', 'Stage', 'Class'],
                               columns='Split', values='Recall')
    pivot['Delta'] = (pivot['test'] - pivot['oof']).round(3)
    pivot['Consistent'] = pivot['Delta'].abs() < 0.10  # 10% tolerance

    return pivot.reset_index()

consistency = oof_test_consistency(data)
display(consistency)

n_inconsistent = (~consistency['Consistent']).sum()
if n_inconsistent == 0:
    print("\n✅ OOF-Test consistency holds for all classes (Δrecall < 0.10)")
else:
    print(f"\n⚠️ OOF-Test inconsistency in {n_inconsistent} classes (Δrecall ≥ 0.10)")
    display(consistency[~consistency['Consistent']])

## 8. Thesis Comparison

Compare M3 results against the thesis error matrix results.

In [ ]:
# Thesis test results (reference)
thesis_results = {
    'CBC_Only': {
        'S1': {'IAS_correct': 154, 'IAS_total': 165,
               'DAS_correct': 43, 'DAS_total': 64},
        'S2': {'HA': (32, 45), 'DEA': (34, 49),
               'Normal': (21, 31), 'HGB_HTZ': (23, 40)}
    },
    'CBC_BIO': {
        'S1': {'IAS_correct': 153, 'IAS_total': 165,
               'DAS_correct': 39, 'DAS_total': 64},
        'S2': {'HA': (37, 45), 'DEA': (39, 49),
               'Normal': (22, 31), 'HGB_HTZ': (29, 40)}
    }
}

print("=== Thesis vs M3 Per-Class Recall Comparison (Test) ===\n")
for scenario in SCENARIOS:
    print(f"--- {scenario} ---")

    # Stage 2
    df = data[scenario][2]['test']
    print(f"  {'Class':<10} {'Thesis Recall':<14} {'M3 Recall':<12} {'Delta':<8}")
    for cls in S2_NAMES:
        thesis_correct, thesis_total = thesis_results[scenario]['S2'][cls]
        thesis_recall = thesis_correct / thesis_total

        cls_id = S2_NAMES.index(cls)
        m3_sub = df[df['y_true'] == cls_id]
        m3_recall = m3_sub['correct'].mean() if len(m3_sub) > 0 else 0

        delta = m3_recall - thesis_recall
        marker = "✅" if abs(delta) < 0.01 else "⚠️"
        print(f"  {DISP_MAP.get(cls, cls):<10} {thesis_recall:<14.3f} {m3_recall:<12.3f} "
              f"{delta:+.3f} {marker}")
    print()

## 9. Manifest Update

## 📋 M3 Summary

### Files Created:

| File | Description |
|---|---|
| `src/m3_errors.py` | Error analysis functions module |
| `per_class_metrics.xlsx` | Per-class P/R/F1 (OOF + Test) |
| `confusion_matrices.xlsx` | All CMs (raw + norm + zone) |
| `misclassification_patterns.xlsx` | Error pair frequencies |
| `clinical_risk_matrix.xlsx` | Clinical impact scoring |
| `scenario_improvement.xlsx` | CBC_Only vs CBC_BIO delta |
| 10+ figures (TIFF+PNG) | Confusion matrix, confidence, feature profile |

### Next Step: M4 and thesis writing will reference these analyses.